# Context Managers and the `with` Statement

Whenever your code interacts with external resources—like opening a file, connecting to a database, or acquiring a thread lock—you are responsible for "cleaning up" that resource when you are done. If you open a file but forget to close it, that file might remain locked by your operating system, or you might cause a memory leak.

Context Managers solve this problem by automatically handling the "setup" and "teardown" phases of a resource, guaranteeing that the cleanup happens even if your code crashes in the middle of the operation.

### 1. The `with` Statement
The `with` statement is the syntax used to trigger a Context Manager. It replaces the clunky `try/finally` blocks found in other languages. When the indented block of code under the `with` statement finishes (or if it throws an error), the Context Manager automatically closes the resource.

### 2. Under the Hood: `__enter__` and `__exit__`
To make your own custom Context Manager, you create a Class that contains two special "dunder" (double underscore) methods:
*   `__enter__(self)`: Runs the moment the `with` block starts. What this method `returns` gets assigned to the variable after the `as` keyword.
*   `__exit__(self, exc_type, exc_val, traceback)`: Runs the exact moment the `with` block ends. The arguments it accepts allow it to inspect any exceptions (errors) that occurred inside the block and decide whether to suppress them or let the program crash.

### 3. The `@contextmanager` Decorator
Writing a full Class with `__enter__` and `__exit__` can be verbose. Python's built-in `contextlib` library provides a brilliant shortcut. By combining a **Decorator** and a **Generator** (the `yield` keyword we learned earlier), you can create a Context Manager in just a few lines of code. The code before the `yield` acts as setup, and the code after acts as teardown.

In [1]:
import time
from contextlib import contextmanager

# ==========================================
# 1. THE CLASSIC USE CASE: FILE I/O
# ==========================================

# ❌ The Old/Risky Way:
file = open("test.txt", "w")
try:
    file.write("Hello, World!")
    # If an error happens here, the file might never close!
finally:
    file.close() 

# ✅ The Pythonic Way (Using a Context Manager):
# The file is automatically closed the moment the block ends.
with open("test.txt", "w") as file:
    file.write("Hello, safely managed World!")

# ==========================================
# 2. BUILDING A CUSTOM CONTEXT MANAGER (Class-Based)
# ==========================================
class DatabaseConnection:
    def __init__(self, db_name):
        self.db_name = db_name

    def __enter__(self):
        print(f"[SETUP] Connecting to database '{self.db_name}'...")
        # What we return here becomes the 'db' variable in the 'with' block
        return self 

    def query(self, sql):
        print(f"   -> Executing query: {sql}")

    def __exit__(self, exc_type, exc_val, traceback):
        print(f"[TEARDOWN] Closing connection to '{self.db_name}'.")
        # If an error occurred inside the 'with' block, exc_type will catch it
        if exc_type:
            print(f"   -> An error occurred: {exc_val}. Safely rolling back changes.")
        # Returning True suppresses the exception. Returning False lets the program crash.
        return True 

print("\n--- Testing Class-Based Context Manager ---")
with DatabaseConnection("UserDB") as db:
    db.query("SELECT * FROM users")
    # Let's trigger an intentional error to see if __exit__ still runs
    raise ValueError("Simulated Database Crash!")

# ==========================================
# 3. BUILDING A CUSTOM CONTEXT MANAGER (Generator-Based)
# ==========================================
# This is the fast, modern way to build simple context managers.
# We will build a 'Timer' to measure how long a block of code takes to run.

@contextmanager
def code_timer(label):
    # Everything BEFORE the yield is the __enter__ logic
    start_time = time.time()
    print(f"\n[TIMER START] {label}")
    
    yield # Pauses here and gives control to the 'with' block
    
    # Everything AFTER the yield is the __exit__ logic
    end_time = time.time()
    print(f"[TIMER END] {label} took {end_time - start_time:.4f} seconds.")

with code_timer("Heavy Computation"):
    # This is the block where the 'yield' pauses
    print("   -> Crunching numbers...")
    time.sleep(0.5) # Simulating heavy work


--- Testing Class-Based Context Manager ---
[SETUP] Connecting to database 'UserDB'...
   -> Executing query: SELECT * FROM users
[TEARDOWN] Closing connection to 'UserDB'.
   -> An error occurred: Simulated Database Crash!. Safely rolling back changes.

[TIMER START] Heavy Computation
   -> Crunching numbers...
[TIMER END] Heavy Computation took 0.5000 seconds.
